# FitVerse Body Measurement Module — Baseline vs. Improved Approach

**Module:** Detect Size (Body Measurement Extraction)
**Based on:** FitVerse: An AI-Powered Fashion Intelligence Platform (Ahmed et al., 2026)
**Purpose of this notebook:** Reproduce the baseline landmark-based measurement approach described in the paper, identify its limitations experimentally, and present an improved segmentation-based approach with a robustness fallback mechanism.

**Structure of this notebook:**
1. Setup — install MediaPipe, load the Pose Landmarker model
2. Baseline Approach — direct landmark-to-landmark distance (as described in the paper)
3. Problem Found #1 — hip measurement is structurally wrong when computed from raw landmarks
4. Improved Approach — segmentation-mask-based width measurement
5. Problem Found #2 — segmentation-based waist detection fails with loose/open garments
6. Solution — hybrid fallback mechanism with confidence scoring
7. Robustness Testing — multiple garment types, documenting two distinct failure modes
8. Final Unified Module — `detect_size()`
9. Summary of Contributions & Limitations


## 1. Setup

In [ ]:
!pip install -q mediapipe

In [ ]:
!wget -O pose_landmarker.task -q https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from google.colab import files
from google.colab.patches import cv2_imshow

base_options = python.BaseOptions(model_asset_path="pose_landmarker.task")
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    output_segmentation_masks=True
)
detector = vision.PoseLandmarker.create_from_options(options)
print("Model loaded successfully")

In [ ]:
# Manual landmark drawing (no dependency on mediapipe.solutions,
# which was removed in recent MediaPipe releases for Python 3.13)

POSE_CONNECTIONS = [
    (11, 12), (11, 13), (13, 15), (12, 14), (14, 16),   # shoulders and arms
    (11, 23), (12, 24), (23, 24),                        # torso
    (23, 25), (25, 27), (27, 29), (27, 31),              # left leg
    (24, 26), (26, 28), (28, 30), (28, 32),              # right leg
    (0, 1), (1, 2), (2, 3), (3, 7),                       # face
    (0, 4), (4, 5), (5, 6), (6, 8),
]

def draw_landmarks_manual(rgb_image, detection_result):
    annotated_image = np.copy(rgb_image)
    h, w, _ = annotated_image.shape
    for pose_landmarks in detection_result.pose_landmarks:
        points = [(int(lm.x * w), int(lm.y * h)) for lm in pose_landmarks]
        for start_idx, end_idx in POSE_CONNECTIONS:
            cv2.line(annotated_image, points[start_idx], points[end_idx], (0, 255, 0), 2)
        for point in points:
            cv2.circle(annotated_image, point, 4, (0, 0, 255), -1)
    return annotated_image

def euclidean_pixel(p1, p2, w, h):
    x1, y1 = p1.x * w, p1.y * h
    x2, y2 = p2.x * w, p2.y * h
    return np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)

## 2. Upload a Test Image (Front View)

Upload a full-body, front-facing photo. Good lighting, plain background, and the whole body visible from head to feet.

In [ ]:
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
print(f"Uploaded: {image_path}")

image = mp.Image.create_from_file(image_path)
detection_result = detector.detect(image)
landmarks = detection_result.pose_landmarks[0]
h, w = image.numpy_view().shape[:2]

print(f"Persons detected: {len(detection_result.pose_landmarks)}")
print(f"Landmarks per person: {len(landmarks)}")

annotated = draw_landmarks_manual(image.numpy_view(), detection_result)
cv2_imshow(cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR))

## 3. Baseline Approach — Direct Landmark Distance (as in the FitVerse paper)

The FitVerse paper (Section 5.3) computes body measurements using the **Euclidean distance
between two body landmarks**, then converts pixel distance to a real-world measurement
using a proportional scaling factor derived from the person's known height.

We reproduce this baseline exactly: for each body dimension, take the straight-line
pixel distance between the two relevant MediaPipe landmarks and scale it to centimeters.

In [ ]:
# Step 1: compute the pixel-to-cm scale factor from the person's real height
real_height_cm = 165  # <-- set this to the actual height of the person in the photo

NOSE = landmarks[0]
LEFT_ANKLE = landmarks[27]
pixel_height = euclidean_pixel(NOSE, LEFT_ANKLE, w, h)
scale = real_height_cm / pixel_height

print(f"scale factor = {scale:.4f} cm/pixel")

In [ ]:
# Step 2: baseline measurements — direct landmark-to-landmark distance
# (this is exactly what the paper's methodology describes)

shoulder_width_px = euclidean_pixel(landmarks[11], landmarks[12], w, h)
hip_width_px       = euclidean_pixel(landmarks[23], landmarks[24], w, h)

shoulder_cm_baseline = round(shoulder_width_px * scale, 1)
hip_cm_baseline       = round(hip_width_px * scale, 1)

print("BASELINE RESULTS (paper methodology)")
print(f"  shoulder_cm = {shoulder_cm_baseline}")
print(f"  hip_cm      = {hip_cm_baseline}")

### Observed Problem

The baseline `hip_cm` result is anatomically implausible — it comes out far too small
compared to `shoulder_cm` (in our test image: **~19.5 cm** vs. a shoulder width of
**~38 cm**, i.e. the hip is reported as roughly *half* the shoulder width, which is not
physically realistic for a standard adult body).

**Root cause:** MediaPipe landmarks 23/24 mark the *internal hip joint centers*
(the anatomical joint), not the *external body contour*. This is a correct landmark for
pose/skeleton estimation, but it systematically underestimates hip **width** when used
directly for clothing-size measurement — the paper's stated methodology does not account
for this distinction.

## 4. Improved Approach — Segmentation-Mask-Based Width Measurement

Instead of relying on the raw distance between two internal landmarks, we use
MediaPipe's **body segmentation mask** to measure the actual external body width at a
given vertical position: at the hip's height, we scan the mask horizontally and measure
the true left-to-right extent of the body silhouette.

In [ ]:
def measure_width_at_height(detection_result, y_normalized, image_shape):
    """Measures the actual body width from the segmentation mask
    at a given normalized vertical position (0 = top, 1 = bottom of image)."""
    mask = detection_result.segmentation_masks[0].numpy_view()
    h, w = mask.shape[:2]
    y_px = int(y_normalized * h)
    row = mask[y_px, :] > 0.5
    if row.sum() == 0:
        return None
    xs = np.where(row)[0]
    return xs.max() - xs.min()

hip_y_normalized = (landmarks[23].y + landmarks[24].y) / 2
hip_width_px_v2 = measure_width_at_height(detection_result, hip_y_normalized, image.numpy_view().shape)
hip_cm_improved = round(hip_width_px_v2 * scale, 1)

print("COMPARISON")
print(f"  hip_cm (baseline, landmark-based)    = {hip_cm_baseline}")
print(f"  hip_cm (improved, segmentation-based) = {hip_cm_improved}")

### Result

The corrected hip measurement is dramatically more realistic (roughly **3x** larger,
and now consistent with the shoulder measurement and human body proportions).
This is a concrete, reproducible correction to a structural flaw in the baseline
methodology — not just a minor accuracy tweak.

## 5. Problem Found #2 — Segmentation Fails for Waist Detection with Loose Garments

The waist is not marked by any MediaPipe landmark at all — the paper's methodology
implies a proportional estimate between shoulder and hip. We first tried a more
anatomically accurate approach: scan the segmentation mask between the shoulder and hip
and find the **narrowest horizontal point** (the natural definition of "waist").

This works well for fitted clothing, but fails predictably with loose garments.

In [ ]:
def find_waist_from_mask(detection_result, landmarks, image_shape, safety_margin=0.15):
    """Finds the narrowest point of the torso silhouette, excluding
    the neck/shoulder region to avoid picking up scarves/collars."""
    mask = detection_result.segmentation_masks[0].numpy_view()
    h, w = mask.shape[:2]

    shoulder_y = int(((landmarks[11].y + landmarks[12].y) / 2) * h)
    hip_y = int(((landmarks[23].y + landmarks[24].y) / 2) * h)
    torso_height = hip_y - shoulder_y

    search_start = shoulder_y + int(torso_height * safety_margin)
    search_end = hip_y - int(torso_height * 0.1)

    min_width = float('inf')
    waist_y = search_start
    waist_x_left, waist_x_right = 0, 0

    for y in range(search_start, search_end):
        row = mask[y, :] > 0.5
        if row.sum() == 0:
            continue
        xs = np.where(row)[0]
        width = xs.max() - xs.min()
        if width < min_width:
            min_width = width
            waist_y = y
            waist_x_left, waist_x_right = xs.min(), xs.max()

    return {
        "waist_y_normalized": waist_y / h,
        "waist_width_px": min_width,
        "left_x": waist_x_left,
        "right_x": waist_x_right
    }

waist_data = find_waist_from_mask(detection_result, landmarks, image.numpy_view().shape)
print(waist_data)

In [ ]:
# Visual check: draw the detected waist line on the image
waist_y_px = int(waist_data["waist_y_normalized"] * h)
debug_image = np.copy(image.numpy_view())
cv2.line(debug_image, (waist_data["left_x"], waist_y_px),
                       (waist_data["right_x"], waist_y_px), (255, 0, 0), 3)
cv2.putText(debug_image, "WAIST?", (waist_data["left_x"], waist_y_px - 10),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)
cv2_imshow(cv2.cvtColor(debug_image, cv2.COLOR_RGB2BGR))

**Observed failure (test case: loose cardigan + scarf/hijab):** the detected
"narrowest point" lands near the neck/collar area, not the actual waist. The scarf
fabric creates a false narrow point that the algorithm mistakes for the waist.
This is a real limitation of naive segmentation-based waist detection with loose
outerwear — a failure mode not addressed in the baseline paper.

## 6. Solution — Hybrid Fallback with Confidence Scoring

Rather than trusting the segmentation result blindly, the system checks whether the
detected waist point falls within an anatomically plausible range of the torso
(35%–65% of the distance between shoulder and hip). If not, it automatically falls
back to a proportional estimate and flags the result as **low confidence**, instead of
silently returning a wrong number.

In [ ]:
def find_waist_hybrid(detection_result, landmarks, image_shape, shoulder_y, hip_y):
    """
    Tries segmentation-based waist detection first. If the result falls outside
    the anatomically plausible range (35%-65% of torso height), falls back to a
    proportional estimate. Handles loose-clothing failure cases safely.
    """
    h, w = image_shape[:2]
    torso_height = hip_y - shoulder_y

    mask_result = find_waist_from_mask(detection_result, landmarks, image_shape)
    relative_position = (mask_result["waist_y_normalized"] * h - shoulder_y) / torso_height

    if 0.35 <= relative_position <= 0.65:
        mask_result["method"] = "segmentation"
        mask_result["confidence"] = "high"
        return mask_result
    else:
        fallback_ratio = 0.55
        waist_y = shoulder_y + int(torso_height * fallback_ratio)
        return {
            "waist_y_normalized": waist_y / h,
            "method": "proportional_fallback",
            "confidence": "low",
            "reason": "segmentation result outside plausible torso range (likely loose garment)"
        }

shoulder_y = int(((landmarks[11].y + landmarks[12].y) / 2) * h)
hip_y = int(((landmarks[23].y + landmarks[24].y) / 2) * h)

waist_hybrid_result = find_waist_hybrid(detection_result, landmarks, image.numpy_view().shape, shoulder_y, hip_y)
print(waist_hybrid_result)

## 7. Robustness Testing Across Multiple Garment Types

We tested the hybrid waist detection on three garment scenarios to characterize its
behavior:

| # | Garment type | Segmentation result | Fallback triggered? | Notes |
|---|---|---|---|---|
| 1 | Loose cardigan + scarf/hijab | `relative_position ≈ 0.15` | Yes | False narrow point at the neck/scarf area |
| 2 | Fitted blouse (untucked) + trousers | `relative_position ≈ 0.15` | Yes | No local minimum at all — width increases monotonically because the untucked blouse drapes past the waist, so the silhouette reflects the garment shape, not the body |
| 3 | Fitted, tucked-in clothing | *(recommended next test)* | Expected: No | Body contour should be directly visible to the segmentation mask |

**Key finding:** these are two *distinct* failure modes, not one repeated bug:
- **Case 1** produces a *false* narrow point (wrong answer with apparent confidence)
- **Case 2** produces *no* narrow point at all (monotonic width, correctly flagged as ambiguous)

Both are correctly caught by the hybrid fallback's plausibility check, which is the
system's main safety net against loose or untucked clothing.

### Diagnostic code used to confirm each finding

The two cells below were used to inspect the raw width profile across the torso and
confirm whether a genuine local minimum exists, and where it falls relative to the
torso height.

In [ ]:
def debug_width_profile(detection_result, landmarks, image_shape, safety_margin=0.15):
    """Prints the width of the body silhouette at every row between the
    shoulder and hip, used to visually/numerically confirm whether a genuine
    waist narrowing exists in the mask."""
    mask = detection_result.segmentation_masks[0].numpy_view()
    h, w = mask.shape[:2]

    shoulder_y = int(((landmarks[11].y + landmarks[12].y) / 2) * h)
    hip_y = int(((landmarks[23].y + landmarks[24].y) / 2) * h)
    torso_height = hip_y - shoulder_y

    search_start = shoulder_y + int(torso_height * safety_margin)
    search_end = hip_y - int(torso_height * 0.1)

    widths = []
    for y in range(search_start, search_end):
        row = mask[y, :] > 0.5
        if row.sum() > 0:
            xs = np.where(row)[0]
            widths.append((y, xs.max() - xs.min()))
    return widths, search_start, search_end, torso_height

widths, s_start, s_end, t_height = debug_width_profile(detection_result, landmarks, image.numpy_view().shape)
print(f"search range: {s_start} to {s_end} (torso_height={t_height})")
print(f"first 5 widths: {widths[:5]}")
print(f"last 5 widths:  {widths[-5:]}")
print(f"min width found at: {min(widths, key=lambda x: x[1])}")

## 8. Final Unified Module — `detect_size()`

All of the above is combined into a single function that takes an image path and the
person's real height, and returns measurements, body shape, recommended size, and a
confidence/method flag for the waist estimation.

In [ ]:
def classify_body_shape(measurements):
    """
    Unified with the recommendation engine's taxonomy (FitStyle_v16 BODY_SHAPE_RULES):
    pear | hourglass | apple | rectangle | inverted_triangle

    Uses shoulder + waist + hip together (not shoulder/hip alone), matching the same
    rules already used elsewhere in the FitStyle AI codebase (server.ts Qwen-vision
    prompt), so all three components now agree on one classification standard.

    Thresholds are in cm, converted from the original inch-based rules
    (rounded: 3in ~ 8cm, 4in ~ 10cm, 8-12in ~ 20-30cm). These are a documented
    starting point, not a calibrated anatomical model -- validate against more
    subjects before production use, same caveat as the waist-height offsets above.
    """
    shoulder = measurements.get("shoulder")
    waist = measurements.get("waist")
    hip = measurements.get("hip")

    if shoulder is None or waist is None or hip is None:
        # Missing a required measurement (e.g. waist fallback failed) -- caller
        # should surface this as low-confidence rather than silently guessing.
        return "unknown"

    shoulder_hip_diff = shoulder - hip
    waist_vs_smaller = min(shoulder, hip) - waist

    # Rectangle: shoulder, waist, and hip are all close together
    if abs(shoulder_hip_diff) < 10 and abs(waist - hip) < 10:
        return "rectangle"

    # Hourglass: shoulders and hips are balanced, waist is clearly smaller than both
    if abs(shoulder_hip_diff) < 8 and waist_vs_smaller >= 15:
        return "hourglass"

    # Apple: waist is close to hip width (fuller midsection, not clearly defined)
    if abs(waist - hip) < 10 and waist_vs_smaller < 15:
        return "apple"

    # Pear: hips clearly wider than shoulders
    if hip - shoulder >= 8:
        return "pear"

    # Inverted triangle: shoulders clearly wider than hips
    if shoulder - hip >= 8:
        return "inverted_triangle"

    # Fallback for edge cases that don't cleanly match any rule above
    return "rectangle"


def map_to_size(measurements):
    # NOTE: placeholder thresholds for demonstration.
    # Replace with a real brand size chart before production use.
    SIZE_CHART = {
        "S": {"shoulder_max": 40, "hip_max": 90},
        "M": {"shoulder_max": 44, "hip_max": 100},
        "L": {"shoulder_max": 48, "hip_max": 110},
    }
    for size, limits in SIZE_CHART.items():
        if measurements.get("shoulder", 999) <= limits["shoulder_max"]:
            return size
    return "XL"


def detect_size(image_path, real_height_cm):
    """
    Takes an image path and the person's real height (cm).
    Returns body measurements, body shape, and a recommended size.

    This is the improved pipeline:
      - shoulder: direct landmark distance (reliable — surface landmark)
      - waist: hybrid segmentation + proportional fallback with confidence flag
      - chest / hip / thigh: segmentation-mask width at the relevant torso height
    """
    image = mp.Image.create_from_file(image_path)
    detection_result = detector.detect(image)

    if not detection_result.pose_landmarks:
        raise ValueError("No person detected in the image")

    landmarks = detection_result.pose_landmarks[0]
    h, w = image.numpy_view().shape[:2]

    pixel_height = euclidean_pixel(landmarks[0], landmarks[27], w, h)
    scale = real_height_cm / pixel_height

    shoulder_px = euclidean_pixel(landmarks[11], landmarks[12], w, h)
    shoulder_cm = round(shoulder_px * scale, 1)

    shoulder_y = int(((landmarks[11].y + landmarks[12].y) / 2) * h)
    hip_y = int(((landmarks[23].y + landmarks[24].y) / 2) * h)
    waist_result = find_waist_hybrid(detection_result, landmarks, image.numpy_view().shape, shoulder_y, hip_y)

    heights = {
        "chest": (landmarks[11].y + landmarks[12].y) / 2 + 0.05,
        "waist": waist_result["waist_y_normalized"],
        "hip": (landmarks[23].y + landmarks[24].y) / 2,
        "thigh": (landmarks[23].y + landmarks[25].y) / 2 + 0.03,
    }

    measurements = {"shoulder": shoulder_cm, "height_cm": real_height_cm}
    for name, y_norm in heights.items():
        width_px = measure_width_at_height(detection_result, y_norm, image.numpy_view().shape)
        measurements[name] = round(width_px * scale, 1) if width_px else None

    body_shape = classify_body_shape(measurements)
    recommended_size = map_to_size(measurements)

    return {
        "measurements": measurements,
        "body_shape": body_shape,
        "recommended_size": recommended_size,
        "waist_confidence": waist_result.get("confidence", "unknown"),
        "waist_method": waist_result.get("method", "unknown")
    }

### Fix: unified body-shape taxonomy (2026-09 update)

`classify_body_shape()` above was rewritten to return the same 5 categories the recommendation engine (`FitStyle_v16`) expects -- `pear`, `hourglass`, `apple`, `rectangle`, `inverted_triangle` -- instead of the previous 4-category set (`athletic`, `triangle`, `rectangle`, `hourglass`), which had no mapping to `pear`/`apple` at all and silently zeroed out the body-shape scoring signal downstream. The new rules use shoulder + waist + hip together, matching the same thresholds already used in the Qwen-vision prompt in `server.ts`, so all three components of the system now agree on one standard.

In [ ]:
result = detect_size(image_path, real_height_cm=165)  # set to the actual height in the photo
print(result)

## 9. Summary of Contributions Over the Baseline (FitVerse Paper)

| Issue | Baseline (paper) behavior | Our improvement |
|---|---|---|
| Hip width | Direct landmark distance → anatomically wrong (~19.5 cm vs. expected ~60 cm) | Segmentation-mask width at hip height → realistic result (~61.9 cm) |
| Waist position | Implied proportional estimate only, no anatomical grounding | Segmentation-based narrowest-point detection when reliable |
| Loose clothing | Not addressed — no failure handling described | Hybrid fallback with plausibility check (35%–65% of torso) and explicit confidence flag |
| Failure transparency | Not addressed | System reports `method` and `confidence` for every waist estimate instead of returning a number silently |

### Limitations (to state clearly to the professor)
- Waist/chest/thigh height offsets (e.g., `+0.05`, `+0.03` of torso height) are currently
  empirically approximated, not derived from a calibrated anatomical model — this should
  be validated against more subjects.
- Absolute height still requires manual input; monocular single-image height inference is
  a well-known ill-posed problem in the literature (confirmed by cross-referencing recent
  research, e.g. Günel et al., ICCV 2019; Jia, 2026) and is proposed as future work rather
  than solved here.
- The segmentation-based approach still fails on loose/untucked garments — the fallback
  mechanism catches this safely, but does not recover the true measurement in those cases.
- `SIZE_CHART` values are placeholders for demonstration and must be replaced with a real
  brand size chart before production use.

### Suggested Future Work
- Multi-view (front + side) depth fusion for more robust waist/chest estimation.
- Garment-type detection to pre-emptively flag when segmentation-based measurement is
  unreliable, instead of only detecting it after the fact.
- Calibration against ground-truth tape measurements from real subjects.